In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
%pip install matplotlib

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement matplotlib (from versions: none)
ERROR: No matching distribution found for matplotlib


# HW02

姓名：邓烨涛  
学号：20234080108

本作业按题目顺序完成。理论题给出推导和解释，编程题提供可运行代码。实验部分运行后会自动输出训练曲线、准确率、梯度范数和 MSE 等结果。


In [5]:
import math
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


ModuleNotFoundError: No module named 'matplotlib'

## 2 机器学习基础

### 2.1 理论题

**1. 单隐层 MLP 的前向表达式**

设输入为 $x \in \mathbb{R}^d$，隐层宽度为 $h$，输出类别数为 $c$。单隐层 MLP 可以写成

$$z_1=W_1x+b_1,$$

$$h=\phi(z_1),$$

$$o=W_2h+b_2.$$

其中 $W_1\in \mathbb{R}^{h\times d}$，$b_1\in\mathbb{R}^{h}$，$W_2\in\mathbb{R}^{c\times h}$，$b_2\in\mathbb{R}^{c}$。如果输入是 mini-batch，$X\in\mathbb{R}^{n\times d}$，常用写法为

$$H=\phi(XW_1+b_1),\quad O=HW_2+b_2,$$

此时 $W_1\in\mathbb{R}^{d\times h}$，$W_2\in\mathbb{R}^{h\times c}$。

**2. Sigmoid 与 tanh 的表达式及导数**

Sigmoid 函数为

$$\sigma(x)=\frac{1}{1+e^{-x}}.$$

它的导数为

$$\sigma'(x)=\frac{e^{-x}}{(1+e^{-x})^2}=\sigma(x)(1-\sigma(x)).$$

双曲正切函数为

$$\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}.$$

它的导数为

$$\tanh'(x)=1-\tanh^2(x).$$

这两个函数在输入绝对值较大时都会饱和，导数接近 0，因此在深层网络中可能导致梯度消失。


### 2.2 编程题：用基础张量操作实现单隐层 MLP

下面使用 Fashion-MNIST。核心网络只用 `torch.matmul`、逐元素运算和手动 SGD 更新实现，不使用 `nn.Linear` 或 `nn.Sequential` 来搭建 MLP。


In [ ]:
batch_size = 256
num_inputs = 28 * 28
num_hiddens = 256
num_outputs = 10

transform = transforms.ToTensor()
train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)
train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_iter = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(len(train_dataset), len(test_dataset))


In [ ]:
def init_mlp_params():
    W1 = torch.normal(0, 0.01, size=(num_inputs, num_hiddens), device=device, requires_grad=True)
    b1 = torch.zeros(num_hiddens, device=device, requires_grad=True)
    W2 = torch.normal(0, 0.01, size=(num_hiddens, num_outputs), device=device, requires_grad=True)
    b2 = torch.zeros(num_outputs, device=device, requires_grad=True)
    return [W1, b1, W2, b2]

def relu(X):
    return torch.clamp(X, min=0.0)

def softmax(X):
    X = X - X.max(dim=1, keepdim=True).values
    exp_X = torch.exp(X)
    return exp_X / exp_X.sum(dim=1, keepdim=True)

def cross_entropy(y_hat, y):
    probs = softmax(y_hat)
    return -torch.log(probs[torch.arange(len(y), device=y.device), y] + 1e-12).mean()

def net(X, params):
    W1, b1, W2, b2 = params
    X = X.reshape((-1, num_inputs))
    H = relu(torch.matmul(X, W1) + b1)
    return torch.matmul(H, W2) + b2

def sgd(params, lr, batch_size):
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad / batch_size
            p.grad.zero_()

def evaluate_accuracy(data_iter, params):
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            pred = net(X, params).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return correct / total


In [ ]:
def train_basic_mlp(num_epochs=10, lr=0.5):
    params = init_mlp_params()
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    for epoch in range(num_epochs):
        loss_sum, correct, total = 0.0, 0, 0
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X, params)
            loss = cross_entropy(y_hat, y)
            loss.backward()
            sgd(params, lr, X.shape[0])
            loss_sum += loss.item() * y.numel()
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.numel()
        history['train_loss'].append(loss_sum / total)
        history['train_acc'].append(correct / total)
        history['test_acc'].append(evaluate_accuracy(test_iter, params))
        print(f"epoch {epoch+1}: loss={history['train_loss'][-1]:.4f}, train_acc={history['train_acc'][-1]:.4f}, test_acc={history['test_acc'][-1]:.4f}")
    return params, history

basic_params, basic_history = train_basic_mlp()


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(basic_history['train_loss'], label='train loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Manual MLP on Fashion-MNIST')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 3 模型选择、正则化与 Dropout

### 3.1 理论题

**训练误差、泛化误差和过拟合**

训练误差是模型在训练集上的错误率或平均损失，它反映模型对已见样本的拟合情况。泛化误差是模型在新样本上的期望误差，实际中通常用验证集或测试集误差来近似。过拟合是训练误差很低，但泛化误差较高的现象，说明模型不仅学到了有效规律，也学到了训练集中的噪声和偶然模式。

常见原因包括模型容量太大、训练数据不足、训练轮数过多、正则化不够等。常见缓解办法包括权重衰减、Dropout、早停、数据增强和减小模型规模。

**K 折交叉验证**

K 折交叉验证先把训练数据分成 K 份，每次取 1 份做验证集，其余 K-1 份做训练集，重复 K 次，使每份数据都做一次验证集。最后把 K 次验证误差取平均作为模型选择依据。确定超参数后，再用完整训练集重新训练模型。它的优点是数据利用更充分，结果更稳定；缺点是训练成本约为普通训练的 K 倍。


### 3.2 编程题：L2 正则化与 Dropout

L2 权重衰减在 SGD 中通过先乘 $(1-\eta\lambda)$ 实现。Dropout 通过随机 mask 把部分隐层输出置零，并除以保留概率，使期望不变。测试阶段不使用 Dropout。


In [ ]:
def dropout_layer(X, dropout):
    if dropout == 0:
        return X
    if dropout == 1:
        return torch.zeros_like(X)
    mask = (torch.rand(X.shape, device=X.device) > dropout).float()
    return mask * X / (1.0 - dropout)

def net_dropout(X, params, dropout=0.0, is_training=True):
    W1, b1, W2, b2 = params
    X = X.reshape((-1, num_inputs))
    H = relu(torch.matmul(X, W1) + b1)
    if is_training:
        H = dropout_layer(H, dropout)
    return torch.matmul(H, W2) + b2

def sgd_l2(params, lr, batch_size, weight_decay=0.0):
    with torch.no_grad():
        for p in params:
            p *= (1 - lr * weight_decay)
            p -= lr * p.grad / batch_size
            p.grad.zero_()

def evaluate_accuracy_dropout(data_iter, params, dropout=0.0):
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            pred = net_dropout(X, params, dropout, is_training=False).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return correct / total

def train_regularized_mlp(label, dropout=0.0, weight_decay=0.0, num_epochs=10, lr=0.5):
    params = init_mlp_params()
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    for epoch in range(num_epochs):
        loss_sum, correct, total = 0.0, 0, 0
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net_dropout(X, params, dropout, is_training=True)
            loss = cross_entropy(y_hat, y)
            loss.backward()
            sgd_l2(params, lr, X.shape[0], weight_decay)
            loss_sum += loss.item() * y.numel()
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.numel()
        history['train_loss'].append(loss_sum / total)
        history['train_acc'].append(correct / total)
        history['test_acc'].append(evaluate_accuracy_dropout(test_iter, params, dropout))
        print(f"{label} epoch {epoch+1}: loss={history['train_loss'][-1]:.4f}, train_acc={history['train_acc'][-1]:.4f}, test_acc={history['test_acc'][-1]:.4f}")
    return history

hist_no_reg = train_regularized_mlp('no regularization', dropout=0.0, weight_decay=0.0)
hist_l2 = train_regularized_mlp('L2', dropout=0.0, weight_decay=1e-4)
hist_dropout = train_regularized_mlp('Dropout', dropout=0.5, weight_decay=0.0)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist_no_reg['train_loss'], label='without regularization')
plt.plot(hist_l2['train_loss'], label='L2')
plt.plot(hist_dropout['train_loss'], label='Dropout')
plt.xlabel('epoch')
plt.ylabel('training loss')
plt.title('Loss Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(hist_no_reg['test_acc'], label='without regularization')
plt.plot(hist_l2['test_acc'], label='L2')
plt.plot(hist_dropout['test_acc'], label='Dropout')
plt.xlabel('epoch')
plt.ylabel('test accuracy')
plt.title('Validation/Test Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


一般来说，不加正则化时训练损失下降更快，但测试集表现可能不够稳定；L2 会抑制权重过大，使模型更平滑；Dropout 会让训练损失略高，因为训练时随机丢弃了部分隐层单元，但通常可以减轻过拟合。


## 4 数值稳定性与深层网络

### 4.1 理论题

深层网络反向传播时，梯度中会出现很多层雅可比矩阵的连乘：

$$\prod_{i=t}^{d-1}\frac{\partial h_{i+1}}{\partial h_i}.$$

如果这些矩阵的尺度大多小于 1，连乘后梯度会快速接近 0，形成梯度消失；如果尺度大多大于 1，连乘后梯度会迅速变大，形成梯度爆炸。

Sigmoid 容易带来梯度消失，因为它的导数最大只有 0.25，进入饱和区后导数接近 0。ReLU 在正半轴导数为 1，不会像 Sigmoid 那样在正半轴饱和，因此能缓解梯度消失。但 ReLU 不能完全避免问题：初始化太大时仍可能梯度爆炸；大量输入落在负半轴时也可能出现死亡 ReLU。


### 4.2 编程题：深层网络中的梯度消失和梯度爆炸

使用 `nn.Sequential` 构建 20 层、隐藏单元数为 256 的深层网络，比较 Sigmoid、ReLU 以及不同初始化方法下的梯度范数。


In [ ]:
def make_deep_net(activation='sigmoid', init='normal1', depth=20, width=256):
    layers = [nn.Flatten(), nn.Linear(num_inputs, width)]
    for _ in range(depth - 1):
        layers.append(nn.Sigmoid() if activation == 'sigmoid' else nn.ReLU())
        layers.append(nn.Linear(width, width))
    layers.append(nn.Sigmoid() if activation == 'sigmoid' else nn.ReLU())
    layers.append(nn.Linear(width, num_outputs))
    model = nn.Sequential(*layers).to(device)

    for m in model:
        if isinstance(m, nn.Linear):
            if init == 'normal1':
                nn.init.normal_(m.weight, mean=0, std=1)
            elif init == 'normal10':
                nn.init.normal_(m.weight, mean=0, std=10)
            elif init == 'xavier':
                nn.init.xavier_uniform_(m.weight)
            elif init == 'kaiming':
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)
    return model

def gradient_norms(model, data_iter):
    criterion = nn.CrossEntropyLoss()
    X, y = next(iter(data_iter))
    X, y = X.to(device), y.to(device)
    model.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    norms = []
    for name, p in model.named_parameters():
        if p.grad is not None and 'weight' in name:
            norms.append(p.grad.norm().item())
    return norms

settings = [
    ('Sigmoid + std=1', 'sigmoid', 'normal1'),
    ('ReLU + std=10', 'relu', 'normal10'),
    ('Xavier + Sigmoid', 'sigmoid', 'xavier'),
    ('Kaiming + ReLU', 'relu', 'kaiming'),
]

grad_results = {}
for title, act, init in settings:
    model = make_deep_net(activation=act, init=init)
    norms = gradient_norms(model, train_iter)
    grad_results[title] = norms
    print(title, 'first=', norms[0], 'last=', norms[-1], 'has_nan=', any(math.isnan(v) for v in norms))


In [ ]:
plt.figure(figsize=(8, 5))
for title, norms in grad_results.items():
    clipped = [min(max(v, 1e-6), 1e3) if not math.isnan(v) else 1e3 for v in norms]
    plt.plot(clipped, marker='o', markersize=3, label=title)
plt.yscale('log')
plt.xlabel('layer index')
plt.ylabel('gradient norm, clipped to [1e-6, 1e3]')
plt.title('Gradient Norms in Deep Networks')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


从结果可以看到，`Sigmoid + std=1` 往往会使靠前层梯度非常小，这是梯度消失；`ReLU + std=10` 因为初始权重太大，可能出现梯度爆炸甚至 NaN。使用 Xavier 或 Kaiming 初始化后，梯度范数通常会稳定很多。实际使用时，Sigmoid/tanh 常配合 Xavier，ReLU 常配合 Kaiming。


## 5 分布偏移

### 5.1 理论题

**协变量偏移（Covariate Shift）** 指训练分布和测试分布的输入边缘分布不同：

$$p(x)\ne q(x),$$

但条件分布相同：

$$p(y|x)=q(y|x).$$

也就是说，样本出现的位置变了，但给定输入以后标签规律没有变。

**标签偏移（Label Shift）** 指训练分布和测试分布的标签先验不同：

$$p(y)\ne q(y),$$

但类条件分布相同：

$$p(x|y)=q(x|y).$$

它们的区别是：协变量偏移主要改变输入分布，标签规则不变；标签偏移主要改变类别比例，每个类别内部的特征分布不变。


### 5.2 编程题：协变量偏移下的一维回归

训练集 $P$ 从 $N(-1,1)$ 采样 1000 个 $x$，测试集 $Q$ 从 $N(2,1)$ 采样 500 个 $x$。标签统一为 $y=2x+\epsilon$，所以 $p(y|x)=q(y|x)$，但 $p(x)\ne q(x)$。


In [ ]:
def make_shift_data(n_train=1000, n_test=500, noise_std=0.5):
    x_train = torch.randn(n_train, 1) - 1.0
    y_train = 2.0 * x_train + noise_std * torch.randn(n_train, 1)
    x_test = torch.randn(n_test, 1) + 2.0
    y_test = 2.0 * x_test + noise_std * torch.randn(n_test, 1)
    return x_train.to(device), y_train.to(device), x_test.to(device), y_test.to(device)

x_train, y_train, x_test, y_test = make_shift_data()

plt.figure(figsize=(6, 4))
plt.hist(x_train.cpu().numpy(), bins=30, alpha=0.6, label='P train')
plt.hist(x_test.cpu().numpy(), bins=30, alpha=0.6, label='Q test')
plt.title('Covariate Shift')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
class LinearRegressionTorch(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    def forward(self, x):
        return self.linear(x)

def train_regression(x, y, weights=None, epochs=500, lr=0.05):
    model = LinearRegressionTorch().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(epochs):
        pred = model(x)
        loss_each = (pred - y).pow(2).squeeze()
        if weights is None:
            loss = loss_each.mean()
        else:
            w = weights.squeeze()
            loss = (w * loss_each).sum() / (w.sum() + 1e-12)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model

def mse(model, x, y):
    with torch.no_grad():
        return ((model(x) - y) ** 2).mean().item()

base_model = train_regression(x_train, y_train)
print('ordinary MSE on P:', mse(base_model, x_train, y_train))
print('ordinary MSE on Q:', mse(base_model, x_test, y_test))
print('ordinary weight, bias:', base_model.linear.weight.item(), base_model.linear.bias.item())


In [ ]:
# ?????P ?? train ? 0?Q ?? test ? 1?
x_domain = torch.cat([x_train, x_test], dim=0)
y_domain = torch.cat([
    torch.zeros(len(x_train), 1, device=device),
    torch.ones(len(x_test), 1, device=device)
], dim=0)

domain_model = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1)).to(device)
optimizer = torch.optim.Adam(domain_model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

for _ in range(1000):
    logits = domain_model(x_domain)
    loss = loss_fn(logits, y_domain)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    p_test = torch.sigmoid(domain_model(x_train))
    p_train = 1 - p_test
    weights = p_test / (p_train + 1e-6)
    weights = weights / weights.mean()
    weights = torch.clamp(weights, 0.05, 20.0)

weighted_model = train_regression(x_train, y_train, weights=weights)
print('weighted MSE on P:', mse(weighted_model, x_train, y_train))
print('weighted MSE on Q:', mse(weighted_model, x_test, y_test))
print('weighted weight, bias:', weighted_model.linear.weight.item(), weighted_model.linear.bias.item())


In [ ]:
plt.figure(figsize=(6, 4))
xs = torch.linspace(-4, 5, 200, device=device).reshape(-1, 1)
with torch.no_grad():
    y_base = base_model(xs).cpu().numpy()
    y_weighted = weighted_model(xs).cpu().numpy()
plt.scatter(x_train.cpu().numpy(), y_train.cpu().numpy(), s=8, alpha=0.25, label='train P')
plt.scatter(x_test.cpu().numpy(), y_test.cpu().numpy(), s=8, alpha=0.25, label='test Q')
plt.plot(xs.cpu().numpy(), y_base, label='ordinary ERM', linewidth=2)
plt.plot(xs.cpu().numpy(), y_weighted, label='importance weighted', linewidth=2)
plt.plot(xs.cpu().numpy(), 2 * xs.cpu().numpy(), '--', label='true function')
plt.title('Regression under Covariate Shift')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


这个实验中真实函数始终是 $y=2x+\epsilon$，所以条件分布没有变，变化主要来自 $x$ 的采样分布。普通训练只最小化训练分布 $P$ 上的误差；重要性加权使用 $w_i\propto P(test|x_i)/P(train|x_i)$，会给更像测试分布的训练样本更大权重，因此更接近测试分布 $Q$ 上的风险。由于这里的真实关系是简单线性关系，普通训练也可能表现不错；但该实验仍然展示了协变量偏移和重要性加权的基本做法。
